# DataLoader

python has two collection data types
* sequence
* iterator

so first lets discuss about the difference between the two

## Sequence
Sequence types have a general concept of a positional element in the sense every element has a corresponding index and every starting index in python starts with 0.

Python has built-in mutable and immutable sequence types.

Strings, tuples are immutable - we can access but not modify the content of the sequence

In [1]:
import torch

In [2]:
example_list = ['cat', 'dog', 'buffalo', 'pig', 'monkey']

**this is an example of a sequence which has both `__len__` and `__getitem__` implemented in it**

In [3]:
example_list[0]

'cat'

In [4]:
example_list.__getitem__(1)

'dog'

In [5]:
example_list

['cat', 'dog', 'buffalo', 'pig', 'monkey']

In [6]:
example_list.__len__()

5

In [7]:
for o in example_list:
    print(o)

cat
dog
buffalo
pig
monkey


## Iterator
We saw how sequence types support iteration by being able to access elements by index. We could even write our custom sequence types by implementing the `__getitem__` method.

But there are some limitations:

items must be numerically indexable, with indexing starting at 0
cannot be used with unordered collections, such as sets
If we think about iterating over a collection, what we really need is a way to request the next item in the collection.

If we can do that, our collection does not require being indexable, nor does it need to be ordered (i.e. we don't need the notion of relative positions of elements in the container).

This is exactly what iterables are in general - they provide a method that returns the "next" element in the collection. This approach works equally well with sequence type collections, as well as unordered collection types such as sets.

In [8]:
example_iterator = {'a', 'b', 'a', 'c', 'd', 'e'}
example_iterator

{'a', 'b', 'c', 'd', 'e'}

iterator's have `__next__` and `__iter__` implemented

if we had to iterate over a iterator we must first create an iterator object then we can use the `__next__` method over it

In [10]:
o = iter(example_iterator)

In [11]:
o.__next__()

'd'

In [12]:
o.__next__()

'b'

In [13]:
o.__next__()

'e'

In [14]:
o.__next__()

'c'

In [15]:
o.__next__()

'a'

In [18]:
# o.__next__()

## Lets implement a custom sequence method

In [19]:
import random
class RandomSeq:
    def __init__(self, length, low=0, up=500):
        self.len = length
        self.values = [random.randint(low, up) for _ in range(length)]
        
    def __getitem__(self, idx):
        return self.values[idx]
    
    def __len__(self): return self.len

In [20]:
o = RandomSeq(10)

In [21]:
o = RandomSeq(10)

In [22]:
o[0]

163

lets create a better representation of the object using the `__repr__` method

now that we can see how the sequence is working we can implement a `__repr__` to see what is inside the object

In [25]:
class RandomSeq:
    def __init__(self, length, low=0, up=500):
        self.len = length
        self.values = [random.randint(low, up) for _ in range(length)]
        
    def __getitem__(self, idx):
        return self.values[idx]
    
    def __len__(self): return self.len

    def __repr__(self): return f'#({len(self.values)}), {self.values}'

In [24]:
o = RandomSeq(10)

Now we can see a better representation for the values inside the sequence object just like a list

In [26]:
o

#(10), [439, 208, 376, 64, 327, 186, 307, 427, 31, 239]

In [27]:
for i in o:
    print(i)

439
208
376
64
327
186
307
427
31
239


Lets implement a custom iterator

In [28]:
class RandomIterator:
    def __init__(self, length, low=0, high=500):
        self.len = length
        self.values = [random.randint(low, high) for _ in range(length)]
        self.start = 0
        self.num = 0
        
    def __len__(self):
        return self.length
        
    def __next__(self):
        try:
            self.num += 1
            return self.values[self.num]
        except IndexError:
            raise StopIteration
        
    def __iter__(self): return self

In [29]:
o = RandomIterator(10)

In [30]:
for i in o:
    print(i)

282
445
44
133
217
487
86
219
101


In [31]:
o.values

[118, 282, 445, 44, 133, 217, 487, 86, 219, 101]

# Dataset

A dataset can either be a 
* sequence object (most cases when the dataset is completely available)
* infinite iterator ( cases of online learning where the input data might not be completely at hand)

lets create synthetic data so that we can create a dataset and a dataloader class

lets create the dependent and independent variables for the data

In [32]:
x = torch.randn(10_000, 15)  # independent variables
x

tensor([[-0.7465,  0.3809, -1.4711,  ...,  2.0306, -0.4894, -1.4346],
        [ 0.7222, -1.0027,  1.0881,  ...,  0.0774, -0.5363,  1.9746],
        [ 0.4504,  0.4423,  0.9903,  ..., -0.2071,  0.3381,  0.9857],
        ...,
        [-0.7609, -0.5027, -0.8375,  ..., -1.3022, -1.5286,  1.0320],
        [-2.2290,  1.5539, -0.1123,  ..., -0.3668,  0.6565, -0.4189],
        [ 1.3751,  0.4678,  2.0196,  ..., -0.1138,  0.1955, -0.3188]])

In [33]:
y = torch.randint(0, 2, (10_000,))   # dependent variables

In [34]:
x.shape, y.shape

(torch.Size([10000, 15]), torch.Size([10000]))

In [35]:
class ToyDataset:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __len__(self): return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [36]:
ds = ToyDataset(x, y)

In [37]:
ds[0]

(tensor([-0.7465,  0.3809, -1.4711, -2.0097, -2.4002, -1.4893,  0.0301,  2.6964,
         -0.4353, -0.9497, -0.0102, -0.7211,  2.0306, -0.4894, -1.4346]),
 tensor(1))

Now that we have created the dataset we can create the dataloader

In [38]:
zip(*[ds[i] for i in range(32)])

In [39]:
class ToyDataLoader:
    def __init__(self, ds, bs=32):
        self.ds = ds
        self.bs = bs
        self.start = 0
        
    def __next__(self):
        end = self.start + self.bs if self.start + self.bs < len(self.ds) else len(self.ds)
        xb, yb = zip(*[ds[i] for i in range(self.start, self.start+self.bs)])
        return torch.stack(xb), torch.stack(yb)
    
    def __iter__(self): return self

    def __len__(self): return self.ds.shape[0]

In [40]:
dl = ToyDataLoader(ds)

Now that we have created a DataLoader we can get a batch of data and check its shape

In [41]:
xb, yb = next(iter(dl))

In [42]:
xb.shape, yb.shape

(torch.Size([32, 15]), torch.Size([32]))

In [43]:
for k, (xb, yb) in enumerate(dl):
    print(k, xb.shape, yb.shape)
    if k > 10: break

0 torch.Size([32, 15]) torch.Size([32])
1 torch.Size([32, 15]) torch.Size([32])
2 torch.Size([32, 15]) torch.Size([32])
3 torch.Size([32, 15]) torch.Size([32])
4 torch.Size([32, 15]) torch.Size([32])
5 torch.Size([32, 15]) torch.Size([32])
6 torch.Size([32, 15]) torch.Size([32])
7 torch.Size([32, 15]) torch.Size([32])
8 torch.Size([32, 15]) torch.Size([32])
9 torch.Size([32, 15]) torch.Size([32])
10 torch.Size([32, 15]) torch.Size([32])
11 torch.Size([32, 15]) torch.Size([32])


### Datasets and Dataloaders in pytorch
Data sets can be thought of as big arrays of data. If the data set is small enough (e.g., MNIST, which has 60,000 28x28 grayscale images), a dataset can be literally represented as an array - or more precisely, as a single pytorch tensor. With one number per pixel, MNIST takes about 200 megabytes of RAM, which fits comfortably into a modern computer.

But larger-scale datasets like ImageNet or Places365 have more than a million higher-resolution full-color images. In these cases, an ordinary python array or pytorch tensor would require more than a terabyte of RAM, which is impractical on most computers.

Instead, we need to load the data from disk (or SSD). Unfortunately, the latency of loading from disk is very slow compared to RAM, so we need to do the loading cleverly if we want to load the data quickly.

To solve the problem, pytorch provides two classes:

torch.utils.data.Dataset - This very simple base class represents an array where the actual data may be slow to fetch, typically because the data is in disk files that require some loading, decoding, or other preprocessing. Pytorch provides a variety of different Dataset subclasses. For example, there is a handy one called ImageFolder that treats a directory tree of image files as an array of classified images.
torch.utils.data.DataLoader - This fancy class wraps a Dataset as a stream of data batches. Behind the scenes it uses a few techniques to feed the data faster. You do not need to subclass DataLoader - its purpose is to make a Dataset speedy.

now lets utilize the pytorch's dataloader class for simplicity as it can have many good features like multithreaded data loading, data shuffling, asyncronous device data loading etc

In [44]:
dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=True)

In [45]:
for k, (xb, yb) in enumerate(dl):
    ...

## Fast Dataset Access using DataLoader
When we use a dataset for training, we will usually run through the whole dataset in batches. We could do this ourselves, by just fetching the images one at a time and grouping them.

But a faster way to iterate through the dataset is to wrap our val_set object in a torch.utils.data.DataLoader object. The val_loader we get can magically pull data out of the Dataset much faster than doing it in the smiple way; the DataLoader class does this by using several threads to load and prefetch the data.

The speedup will depend on the system and the number of threads you use (the number of threads to use is specified using num_workers). In practice using DataLoader will typically be 5-20 times faster than direct Dataset access.


`Exercise`

Try adjusting num_workers down to 1 and up to 100. How does this affect the speed?

Try changing batch_size down to 1 or up to 1000.

`*Note*`: the speed differences you see will depend on the specifics of your system setup. If you are running on Google Colab, you may not see much of a speedup from DataLoader. This is because Colab provides a very low-latency virtual disk (so direct Dataset access is faster than on a regular computer), and a virtual CPU with very slow concurrency (so DataLoader multithreading is slower than normal).

#### Other common dataloader tricks. DataLoader can do a few more useful things.

* Although a DataLoader does not put batches on the GPU directly (because of multithreading limitations), it can put the batch in pinned memory, which is faster to copy to the GPU later after you get it out of the DataLoader. Make the DataLoader with pin_memory=True for this.
* During training you usually do not want the batches in same order for every epoch which will lead to improper learning of the model. The DataLoader can shuffle the batches so that they are randomized, instead of sequential. shuffle=True for this.
